In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import sys
from pathlib import Path

# Поменяй путь под свою папку на Google Drive.
# В этой папке должны лежать: dataset.py, model.py, trainer.py.
PROJECT_DIR = Path('/content/drive/MyDrive/ODS_NLP_COURSE_FINAL_PROJECT/LLM_Inference')
DATA_DIR = PROJECT_DIR / 'data' / 'gold_labeled_data'

os.chdir(PROJECT_DIR)

In [3]:
import gc
import json
import random
import warnings
from datetime import datetime
from typing import Optional
import re
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from jinja2 import Template
from tqdm.auto import tqdm


warnings.filterwarnings('ignore')

RANDOM_SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA memory, GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Device: cuda
GPU: NVIDIA L4
CUDA memory, GB: 22.03


In [22]:
# !pip install -q bert-score

# Грузим датасеты

In [4]:
FULL_DATA_XLSX = DATA_DIR / 'app_reviews_ru_gold_labeled.xlsx'
TRAIN_XLSX = DATA_DIR / 'train.xlsx'
TEST_XLSX = DATA_DIR / 'test.xlsx'

DROP_COLS = ['Unnamed: 0', 'Unnamed: 0.1', 'lable_silver_common_AI', 'summary_silver_GPT']


def load_review_data():
    if TRAIN_XLSX.exists() and TEST_XLSX.exists():
        print('Loading existing train.xlsx/test.xlsx')
        train_df = pd.read_excel(TRAIN_XLSX)
        test_df = pd.read_excel(TEST_XLSX)
    else:
        print('train.xlsx/test.xlsx were not found, creating stratified split from FINAL_GOLD')
        df = pd.read_excel(FULL_DATA_XLSX, sheet_name='FINAL_GOLD')
        df = df.drop(columns=DROP_COLS, errors='ignore')
        df = df.dropna(subset=['text', 'label_gold']).reset_index(drop=True)

        train_df, test_df = train_test_split(
            df,
            test_size=0.15,
            random_state=RANDOM_SEED,
            stratify=df['label_gold']
        )

        train_df = train_df.reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)
        train_df.to_excel(TRAIN_XLSX, index=False)
        test_df.to_excel(TEST_XLSX, index=False)

    train_df = train_df.drop(columns=DROP_COLS, errors='ignore')
    test_df = test_df.drop(columns=DROP_COLS, errors='ignore')

    for df in [train_df, test_df]:
        df['text'] = df['text'].fillna('').astype(str)
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
        df['thumbs_up_count'] = pd.to_numeric(df['thumbs_up_count'], errors='coerce')

    return train_df, test_df

train_data, test_data = load_review_data()

print('Train:', train_data.shape)
print('Test:', test_data.shape)
print('Train columns:', train_data.columns.tolist())
print('Class distribution:')
print(train_data['label_gold'].value_counts(normalize=True).round(4))
train_data.head()

Loading existing train.xlsx/test.xlsx
Train: (2714, 6)
Test: (480, 6)
Train columns: ['app_name', 'text', 'rating', 'thumbs_up_count', 'label_gold', 'summary_gold']
Class distribution:
label_gold
user_experience    0.4749
bug_report         0.2598
rating             0.2119
feature_request    0.0534
Name: proportion, dtype: float64


,app_name,text,rating,thumbs_up_count,label_gold,summary_gold
0,Avito,программа Авито очень помогает в продаже б.у т...,5,0,user_experience,Положительный опыт использования приложения
1,VK,Скачала и пользовалась с 2015 года. Новый отзы...,1,24,user_experience,Недовольство модерацией и поддержкой
2,Yandex Maps,Сотрудники Яндекс Поддержки продались и сотруд...,1,5,user_experience,Мешает навязчивая реклама
3,Telegram,"прошу, сделайте так чтоб подарки можно было пр...",5,0,feature_request,Разрешить продажу подарков в любое время
4,Yandex Maps,настройте приложение опять заедает и вылетает,1,3,bug_report,Приложение вылетает


# System prompt

In [5]:
system_prompt = Template(
"""
Ты — NLP-ассистент для анализа русскоязычных отзывов на приложения Google Play.

Тебе будет передан один отзыв в формате:
text: текст отзыва
rating: оценка приложения от 1 до 5
thumbs_up_count: количество лайков отзыва

Твоя задача:
1. Определить основной класс отзыва строго из списка:
- bug_report
- feature_request
- user_experience
- rating

Правила разметки:
- bug_report: пользователь сообщает об ошибке, сбое, зависании, некорректной работе, потере данных.
- feature_request: пользователь просит добавить или изменить функцию. Если одновременно есть явная ошибка и просьба, выбери bug_report.
- user_experience: пользователь описывает опыт использования, удобство, неудобство, плюсы/минусы без явного бага и без просьбы новой функции.
- rating: короткая общая оценка без полезной конкретики.

Приоритет при смешанных случаях:
bug_report > feature_request > user_experience > rating

2. Сформировать summary:
- одна короткая фраза на русском;
- не более 12 слов;
- без выдуманных деталей;
- только главная мысль из отзыва.

Верни только валидный JSON без markdown и пояснений:
{"label": "bug_report|feature_request|user_experience|rating", "summary": "короткое summary"}

Примеры:
Отзыв:
text: все приложение не работает вообще, не заходит и не грузит
rating: 1
thumbs_up_count: 0
Ответ:
{"label": "bug_report", "summary": "Приложение не запускается и не загружается"}

Отзыв:
text: отлично приложение!!!
rating: 5
thumbs_up_count: 0
Ответ:
{"label": "rating", "summary": "Краткая положительная оценка без деталей"}

Отзыв:
text: прошу добавить темную тему
rating: 4
thumbs_up_count: 2
Ответ:
{"label": "feature_request", "summary": "Просьба добавить темную тему"}
"""
)


In [7]:
# Утилиты для инференса LLM

VALID_LABELS = {"bug_report", "feature_request", "user_experience", "rating"}


def format_review(row_or_text, rating=None, thumbs_up_count=None) -> str:
    """
    Приводит отзыв к тому формату, который описан в system_prompt.
    Можно передать либо строку, либо строку DataFrame.
    """
    if isinstance(row_or_text, pd.Series):
        text = str(row_or_text.get("text", ""))
        rating = row_or_text.get("rating", "")
        thumbs_up_count = row_or_text.get("thumbs_up_count", "")
    else:
        text = str(row_or_text)

    return (
        f"text: {text}\n"
        f"rating: {rating if rating is not None else ''}\n"
        f"thumbs_up_count: {thumbs_up_count if thumbs_up_count is not None else ''}"
    )


def prepare_messages(review_text: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": system_prompt.render()},
        {"role": "user", "content": review_text},
    ]


def generate_response(model, tokenizer, messages, max_new_tokens=96):
    """
    Важно:
    - max_new_tokens маленький, потому что ответ должен быть коротким JSON.
    - add_generation_prompt=True нужен для chat-моделей.
    - return_dict=True гарантирует, что inputs будет dict с input_ids/attention_mask.
    """
    model.eval()

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,   # для Qwen thinking-моделей; если модель не поддерживает, строку можно убрать
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            use_cache=True,
        )

    input_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()


def parse_llm_json(response: str) -> dict:
    """
    Пытается достать JSON из ответа модели.
    Если модель добавила лишний текст, вырезаем первый {...}.
    """
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
    response = response.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*?\}", response, flags=re.DOTALL)
    if match:
        response = match.group(0)

    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        return {"label": "parse_error", "summary": response[:120], "raw_response": response}

    label = str(data.get("label", "")).strip()
    summary = str(data.get("summary", "")).strip()

    if label not in VALID_LABELS:
        return {"label": "parse_error", "summary": summary, "raw_response": response}

    # Жёстко ограничим summary на случай, если модель разговорилась
    summary = " ".join(summary.split()[:12])

    return {"label": label, "summary": summary, "raw_response": response}


In [8]:
def run_agent(row_or_text, model, tokenizer, system_prompt: Template, debug: bool = False):
    """
    Один отзыв -> один вызов model.generate -> dict с label и summary.
    Старый while True здесь не нужен, потому что tools/function calling не используются.
    """
    review_text = format_review(row_or_text)
    messages = prepare_messages(review_text)

    if debug:
        print("USER MESSAGE:")
        print(messages[-1]["content"])

    response = generate_response(
        model=model,
        tokenizer=tokenizer,
        messages=messages,
        max_new_tokens=96,
    )

    if debug:
        print("\nRAW MODEL RESPONSE:")
        print(response)

    return parse_llm_json(response)


In [9]:
# %pip install --upgrade transformers

In [10]:
# Load model and tokenizer

model_name = "Qwen/Qwen3.5-4B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,      # для T4 лучше float16; на A100/H100 можно пробовать bfloat16
    device_map="auto",
    trust_remote_code=True,
)

model.eval()


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 2560)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (in_proj_qkv): Linear(in_features=2560, out_features=8192, bias=False)
          (in_proj_z): Linear(in_features=2560, out_features=4096, bias=False)
          (in_proj_b): Linear(in_features=2560, out_features=32, bias=False)
          (in_proj_a): Linear(in_features=2560, out_features=32, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (down_proj): Linear(

In [11]:
test_data

,app_name,text,rating,thumbs_up_count,label_gold,summary_gold
0,VK,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями
1,Telegram,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно
2,VK,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит
3,WhatsApp,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно
4,VK,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения
...,...,...,...,...,...,...
475,VK,самое худшее приложение.,1,1,rating,Краткая негативная оценка без деталей
476,WhatsApp,Приложение вылетает при запуске. Mi6,1,7,bug_report,Приложение вылетает
477,VK,очень нравится приложение,5,0,rating,Краткая положительная оценка без деталей
478,Telegram,"не грузят гс, видео, фото, кружки ни с впн, ни...",4,0,user_experience,Проблемы с загрузкой медиа


In [12]:
test_row = test_data.iloc[0]
test_mess = format_review(test_row)
print(test_mess)


text: с каждым новым обновлением всё хуже, даже сообщения не доходят до меня, даже посмотреть их не могу
rating: 1
thumbs_up_count: 0


In [13]:
test_mess

'text: с каждым новым обновлением всё хуже, даже сообщения не доходят до меня, даже посмотреть их не могу\nrating: 1\nthumbs_up_count: 0'

In [14]:
res = run_agent(test_mess, model, tokenizer, system_prompt, debug=True)
res


USER MESSAGE:
text: text: с каждым новым обновлением всё хуже, даже сообщения не доходят до меня, даже посмотреть их не могу
rating: 1
thumbs_up_count: 0
rating: 
thumbs_up_count: 

RAW MODEL RESPONSE:
{"label": "bug_report", "summary": "Сообщения не доходят после обновлений"}


{'label': 'bug_report',
 'summary': 'Сообщения не доходят после обновлений',
 'raw_response': '{"label": "bug_report", "summary": "Сообщения не доходят после обновлений"}'}

In [15]:
res

{'label': 'bug_report',
 'summary': 'Сообщения не доходят после обновлений',
 'raw_response': '{"label": "bug_report", "summary": "Сообщения не доходят после обновлений"}'}

In [16]:
# Инференс по всей тестовой выборке + classification_report

pred_rows = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data)):
    pred = run_agent(row, model, tokenizer, system_prompt, debug=False)

    pred_rows.append({
        "id": idx,
        "text": row["text"],
        "rating": row.get("rating", None),
        "thumbs_up_count": row.get("thumbs_up_count", None),
        "label_gold": row.get("label_gold", None),
        "summary_gold": row.get("summary_gold", None),
        "label_pred": pred["label"],
        "summary_pred": pred["summary"],
        "raw_response": pred.get("raw_response", ""),
    })

res_df = pd.DataFrame(pred_rows)

# Отдельно посмотрим ошибки парсинга, если они есть
print("Parse errors:", (res_df["label_pred"] == "parse_error").sum())

# Метрики по классификации
valid_eval = res_df[res_df["label_pred"].isin(sorted(VALID_LABELS))].copy()

print(classification_report(
    valid_eval["label_gold"],
    valid_eval["label_pred"],
    labels=["bug_report", "feature_request", "user_experience", "rating"],
    digits=4,
    zero_division=0,
))

res_df.to_csv("llm_inference_class_summary_predictions.csv", index=False)
res_df.head()


  0%|          | 0/480 [00:00<?, ?it/s]

Parse errors: 0
                 precision    recall  f1-score   support

     bug_report     0.5812    0.8880    0.7025       125
feature_request     0.6190    0.5000    0.5532        26
user_experience     0.7513    0.6491    0.6965       228
         rating     0.9859    0.6931    0.8140       101

       accuracy                         0.7125       480
      macro avg     0.7343    0.6825    0.6915       480
   weighted avg     0.7492    0.7125    0.7150       480



,id,text,rating,thumbs_up_count,label_gold,summary_gold,label_pred,summary_pred,raw_response
0,0,"с каждым новым обновлением всё хуже, даже сооб...",1,0,user_experience,Проблемы с чатами и сообщениями,bug_report,Сообщения не доходят после обновлений,"{""label"": ""bug_report"", ""summary"": ""Сообщения ..."
1,1,"приложение супер, только почему-то после того ...",5,0,user_experience,Приложение работает некорректно,bug_report,Ошибка: после удаления бесплатных историй нель...,"{""label"": ""bug_report"", ""summary"": ""Ошибка: по..."
2,2,при включении музыки постоянно виснит и вылета...,1,1,bug_report,Приложение лагает и тормозит,bug_report,Приложение зависает и вылетает при включении м...,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
3,3,после обновления приложения не работает не мог...,1,0,bug_report,Приложение работает некорректно,bug_report,Приложение перестало работать после обновления,"{""label"": ""bug_report"", ""summary"": ""Приложение..."
4,4,очень неудобное приложение. проблем внутри куч...,1,0,feature_request,Предложение по улучшению приложения,user_experience,Критика неудобства и проблем с использованием ...,"{""label"": ""user_experience"", ""summary"": ""Крити..."


In [18]:
from sklearn.preprocessing import LabelEncoder

In [19]:
le = LabelEncoder()
res_df['label_real_num'] = le.fit_transform(res_df['label_gold'])
res_df['label_pred_num'] = le.fit_transform(res_df['label_pred'])

In [20]:
print(classification_report(res_df['label_real_num'], res_df['label_pred_num']))

              precision    recall  f1-score   support

           0       0.58      0.89      0.70       125
           1       0.62      0.50      0.55        26
           2       0.99      0.69      0.81       101
           3       0.75      0.65      0.70       228

    accuracy                           0.71       480
   macro avg       0.73      0.68      0.69       480
weighted avg       0.75      0.71      0.72       480



In [23]:
import re
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from bert_score import score as bert_score


def normalize_text(text):
    """
    Нормализация текста summary:
    - приводим к строке;
    - убираем лишние пробелы;
    - приводим к нижнему регистру;
    - заменяем ё на е.
    """
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()
    text = text.replace("ё", "е")
    text = re.sub(r"\s+", " ", text)

    return text


def tokenize_ru(text):
    """
    Простая токенизация для русскоязычного текста.
    Берем русские/английские слова и числа.
    """
    text = normalize_text(text)
    return re.findall(r"[а-яa-z0-9]+", text)


def lcs_length(tokens_a, tokens_b):
    """
    Длина наибольшей общей подпоследовательности.
    Нужна для ROUGE-L.
    """
    n = len(tokens_a)
    m = len(tokens_b)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n):
        for j in range(m):
            if tokens_a[i] == tokens_b[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[n][m]


def rouge_l_score(reference, prediction):
    """
    Считает ROUGE-L Precision, Recall и F1 для одной пары:
    reference = summary_gold
    prediction = summary_pred
    """
    ref_tokens = tokenize_ru(reference)
    pred_tokens = tokenize_ru(prediction)

    if len(ref_tokens) == 0 and len(pred_tokens) == 0:
        return 1.0, 1.0, 1.0

    if len(ref_tokens) == 0 or len(pred_tokens) == 0:
        return 0.0, 0.0, 0.0

    lcs = lcs_length(ref_tokens, pred_tokens)

    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


def evaluate_summaries(
    df,
    gold_col="summary_gold",
    pred_col="summary_pred",
    label_col=None,
    bert_model="xlm-roberta-base",
    batch_size=16
):
    """
    Оценивает качество summary по:
    - ROUGE-L;
    - BERTScore.

    Возвращает:
    - detailed_df: исходный датафрейм + метрики для каждой строки;
    - overall_metrics: общие средние метрики;
    - by_label_metrics: метрики по классам, если передан label_col.
    """

    detailed_df = df.copy()

    detailed_df[gold_col] = detailed_df[gold_col].fillna("").astype(str)
    detailed_df[pred_col] = detailed_df[pred_col].fillna("").astype(str)

    # ---------- ROUGE-L ----------
    rouge_p_list = []
    rouge_r_list = []
    rouge_f1_list = []

    for gold, pred in tqdm(
        zip(detailed_df[gold_col], detailed_df[pred_col]),
        total=len(detailed_df),
        desc="Calculating ROUGE-L"
    ):
        p, r, f1 = rouge_l_score(gold, pred)
        rouge_p_list.append(p)
        rouge_r_list.append(r)
        rouge_f1_list.append(f1)

    detailed_df["rougeL_precision"] = rouge_p_list
    detailed_df["rougeL_recall"] = rouge_r_list
    detailed_df["rougeL_f1"] = rouge_f1_list

    # ---------- BERTScore ----------
    gold_texts = detailed_df[gold_col].apply(normalize_text).tolist()
    pred_texts = detailed_df[pred_col].apply(normalize_text).tolist()

    valid_mask = [
        len(gold.strip()) > 0 and len(pred.strip()) > 0
        for gold, pred in zip(gold_texts, pred_texts)
    ]

    detailed_df["bertscore_precision"] = 0.0
    detailed_df["bertscore_recall"] = 0.0
    detailed_df["bertscore_f1"] = 0.0

    valid_gold = [gold for gold, is_valid in zip(gold_texts, valid_mask) if is_valid]
    valid_pred = [pred for pred, is_valid in zip(pred_texts, valid_mask) if is_valid]

    if len(valid_gold) > 0:
        device = "cuda" if torch.cuda.is_available() else "cpu"

        P, R, F1 = bert_score(
            cands=valid_pred,
            refs=valid_gold,
            model_type=bert_model,
            batch_size=batch_size,
            device=device,
            verbose=True
        )

        valid_indices = detailed_df.index[valid_mask]

        detailed_df.loc[valid_indices, "bertscore_precision"] = P.cpu().numpy()
        detailed_df.loc[valid_indices, "bertscore_recall"] = R.cpu().numpy()
        detailed_df.loc[valid_indices, "bertscore_f1"] = F1.cpu().numpy()

    # ---------- Общие метрики ----------
    metric_cols = [
        "rougeL_precision",
        "rougeL_recall",
        "rougeL_f1",
        "bertscore_precision",
        "bertscore_recall",
        "bertscore_f1"
    ]

    overall_metrics = detailed_df[metric_cols].mean().to_frame("mean_value")

    # ---------- Метрики по классам ----------
    by_label_metrics = None

    if label_col is not None and label_col in detailed_df.columns:
        by_label_metrics = (
            detailed_df
            .groupby(label_col)[metric_cols]
            .mean()
            .sort_values("bertscore_f1", ascending=False)
        )

    return detailed_df, overall_metrics, by_label_metrics

In [24]:
summary_eval_df, summary_overall_metrics, _ = evaluate_summaries(
    res_df,
    gold_col="summary_gold",
    pred_col="summary_pred",
    bert_model="xlm-roberta-base",
    batch_size=16
)

summary_overall_metrics

Calculating ROUGE-L:   0%|          | 0/480 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

  0%|          | 0/33 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/30 [00:00<?, ?it/s]

done in 0.84 seconds, 572.53 sentences/sec


,mean_value
rougeL_precision,0.216056
rougeL_recall,0.264383
rougeL_f1,0.230383
bertscore_precision,0.857979
bertscore_recall,0.871277
bertscore_f1,0.864381


In [25]:
summary_eval_df[
    ["text", "summary_gold", "summary_pred", "rougeL_f1", "bertscore_f1"]
].sort_values("bertscore_f1").head(20)

,text,summary_gold,summary_pred,rougeL_f1,bertscore_f1
124,Приложение просто замечательнО! Но есть не бол...,Добавить галочки прочтения сообщений,Пользователь описывает плюсы и минусы использо...,0.00,0.771957
96,"Вроде оф приложения, но такое не удобное. Ладн...",Добавить настройки сообщений,"Множество багов: ошибки доступа, зависания, пр...",0.00,0.783056
285,"В поисковом запросе, не понятно в каком городе...",Некорректно работает поиск,Предложения по улучшению интерфейса поиска и о...,0.00,0.786255
451,что за проблемы со входом????в день по два раз...,Приложение вылетает,Постоянные сбои входа и навязчивая реклама,0.00,0.788978
389,"подключена подписка на музыку, а реклама в вид...",Недовольство платными функциями,Реклама показывается несмотря на подписку,0.00,0.793016
457,классное приложение мне нравиться,Краткая положительная оценка без деталей,Пользователь доволен приложением,0.00,0.793216
461,"Всё хорошо, спасибо.",Краткая положительная оценка без деталей,Пользователь доволен приложением,0.00,0.793216
119,әуе қорғанысы а🥹,Краткая положительная оценка без деталей,Недовольство отсутствием защиты от ветра,0.00,0.794710
187,При использовании в режиме карты гибрид показы...,Проблемы с загрузкой данных,Сбой отображения карты и дорог без интернета,0.00,0.795387
154,вв не можете исправить вк музыку? смысл брать ...,Недовольство платными функциями,"Приложение лагает, музыка пропадает, подписка ...",0.00,0.796651


In [26]:
summary_eval_df[
    ["text", "summary_gold", "summary_pred", "rougeL_f1", "bertscore_f1"]
].sort_values("bertscore_f1", ascending=False).head(20)

,text,summary_gold,summary_pred,rougeL_f1,bertscore_f1
60,все отлично 👍,Краткая положительная оценка без деталей,Краткая положительная оценка без деталей,1.0,1.0
83,Отстой минус 5 звезд,Краткая негативная оценка без деталей,Краткая негативная оценка без деталей,1.0,1.0
362,Самое плохое приложение,Краткая негативная оценка без деталей,Краткая негативная оценка без деталей,1.0,1.0
41,гугл лучше,Краткая негативная оценка без деталей,Краткая негативная оценка без деталей,1.0,1.0
193,Пашка Дуров круче,Краткая негативная оценка без деталей,Краткая негативная оценка без деталей,1.0,1.0
213,ужас,Краткая негативная оценка без деталей,Краткая негативная оценка без деталей,1.0,1.0
221,ГОВТО,Краткая негативная оценка без деталей,Краткая негативная оценка без деталей,1.0,1.0
475,самое худшее приложение.,Краткая негативная оценка без деталей,Краткая негативная оценка без деталей,1.0,1.0
40,говорю коротко и ясно:макс альтушка а телеграм...,Краткая положительная оценка без деталей,Краткая положительная оценка без деталей,1.0,1.0
69,отличное,Краткая положительная оценка без деталей,Краткая положительная оценка без деталей,1.0,1.0


In [28]:
summary_eval_df[
    ["text", "summary_gold", "summary_pred", "rougeL_f1", "bertscore_f1"]
][summary_eval_df["rougeL_f1"].between(0.6, 0.8)]

,text,summary_gold,summary_pred,rougeL_f1,bertscore_f1
5,xoрушия,Краткая положительная оценка без деталей,Краткая положительная оценка,0.750000,0.959779
16,разрабы сделайте так чтоб на прокси нормально ...,Улучшить работу прокси,Просьба улучшить работу приложения через прокси,0.666667,0.907164
39,Приложение после обновления разочаровало из-за...,вернуть выбор способа передвижения,Пользователь просит вернуть выбор способа пере...,0.800000,0.944244
48,высший класс,Краткая положительная оценка без деталей,Высокая оценка без дополнительных деталей,0.600000,0.903824
79,вылетает постоянно,Приложение вылетает,Приложение постоянно вылетает,0.800000,0.972407
97,Не работает,Приложение работает некорректно,Приложение не работает,0.666667,0.911029
248,Как вы надоели. Прошло столько времени с багом...,Проблемы с отображением объявлений,Баг с не отображением активных объявлений,0.600000,0.939950
250,ОФЛАЙН КАРТЫ НЕ РАБОТАЮТ !!! Скачаны 1200Мб мо...,ОФЛАЙН КАРТЫ НЕ РАБОТАЮТ,"Офлайн карты не загружаются, белый экран",0.600000,0.910394
292,вообще не работает,Приложение работает некорректно,Приложение не работает,0.666667,0.911029
327,"Павел Дуров, здравствуйте, сделайте пожалуйста...",Добавить видео в профиль,Просьба добавить возможность вставки видео в п...,0.727273,0.930951
